In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Parameters
n_obs   = 10          # Number of independent N(0,1) variables in each sample
n_sims  = 50_000      # Number of simulations
c       = 4.685       # Common tuning constant for Tukey bi-square

def tukey_weights(r, c):
    """Tukey bi-square weight function, gives weight only for |r|<=c"""
    mask = np.abs(r) <= c
    w = np.zeros_like(r)
    t = r[mask] / c
    w[mask] = (1 - t**2)**2          # (1 - (r/c)^2)^2
    return w

# --- 1) Classical chi-square statistic Σ x_i² --------------------------------------------
chi2_stats = np.empty(n_sims)

# --- 2) Weighted statistic Σ w_i x_i² ------------------------------------------
weighted_stats = np.empty(n_sims)

for k in range(n_sims):
    x = np.random.randn(n_obs)               # N(0,1)
    chi2_stats[k] = np.sum(x**2)             # Theoretical ~ χ²(n_obs)

    w  = tukey_weights(x, c)                 # Tukey weights
    weighted_stats[k] = np.sum(w * x**2)     # No longer χ²

# --------------------- KS test --------------------------------------------
d_chi2,  p_chi2  = stats.kstest(chi2_stats,
                                'chi2', args=(n_obs,))
d_tukey, p_tukey = stats.kstest(weighted_stats,
                                'chi2', args=(n_obs,))

print(f"Σ x_i² vs χ²({n_obs})  : KS p-value = {p_chi2:.3f}")
print(f"Tukey-weighted vs χ²({n_obs}): KS p-value = {p_tukey:.3e}")

# --------------------- Visual histograms -----------------------------------------
fig1 = plt.figure()
plt.hist(chi2_stats, 100, density=True, label='simulated')
xgrid = np.linspace(0, chi2_stats.max(), 400)
plt.plot(xgrid, stats.chi2.pdf(xgrid, df=n_obs),
         lw=2, label='χ² pdf')
plt.title("Σ x_i²    (should follow χ²)")
plt.legend()

fig2 = plt.figure()
plt.hist(weighted_stats, 100, density=True, label='simulated')
plt.plot(xgrid, stats.chi2.pdf(xgrid, df=n_obs),
         lw=2, label='χ² pdf')
plt.title("Tukey-weighted Σ w_i x_i²    (clearly deviates from χ²)")
plt.legend()

plt.show()